##Avvikanden

Ledningen önskar få svar på frågan: Eventuella avvikelser – något oväntat mönster som sticker ut?

Frågeställningen är hur jag kan nå fram till det som sticker ut. I min del av data analysen kommer jag försöka hitta avvikande information. 

Jag kommer att börja med att ta fram kortfattade data information av varje kategori.

Sedan kommer jag fokusera på följande:
- tid
- antal ordrar
- städer
- kategorier

Med ovan nämnda fokusområden (samma som kategorierna i data) kommer jag ställa dem mot varandra och se om något sticker ut. Jag kommer fokusera på onormala toppar och bottnar i datan.

Med hjälp av piecharts och stapeldiagram kommer jag visa det jag finner. Mina andra kollegors uppgifter kommer ge en kompletterande bild av hela data analysen.

I början behöver vi inportera numpy, matplotlib och pandas. För att enkelt kunna tolka data genom grafik och i kodblock.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv("../data/ecommerce_sales.csv")

df.info()
df["city"].value_counts()


In [ ]:
df["category"].value_counts()


In [ ]:
df["price"].value_counts()


In [ ]:
df["units"].value_counts()


In [ ]:
df["revenue"].value_counts()

In [ ]:
df["date"].value_counts()

Ovan har jag tagit fram antal från varje kategori. nedan kommer jag att försöka förtydliga och tolka datan

In [ ]:
# vad har sålts i varje stad, när och hur mycket.

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")

def show_month_categories_city(df, month):

    month_names = {
        "2024-01": "January 2024",
        "2024-02": "February 2024",
        "2024-03": "March 2024",
        "2024-04": "April 2024",
        "2024-05": "May 2024",
        "2024-06": "June 2024",
        }

    month_data = (
        df[df["month"] == month]
        .groupby(["city", "category"])["units"]
        .sum()
        .reset_index()
    )

    visning = month_data.pivot(
        index="category", 
        columns="city", 
        values="units"
    )

    print(f"{month_names[month]} Sales by City and Category")
    print(visning)

show_month_categories_city(df, "2024-01")


In [ ]:
show_month_categories_city(df, "2024-02")

In [ ]:
show_month_categories_city(df, "2024-03")

In [ ]:
show_month_categories_city(df, "2024-04")

In [ ]:
show_month_categories_city(df, "2024-05")

In [ ]:
show_month_categories_city(df, "2024-06")

In [ ]:
#upptäckt nan i datan ovan specificerar här

df.isna().sum() #ger resultat 0 på samtliga så det är komplett på kategorierna från org data
print(f"Checking for missing values in dataset: {df.isna().sum().sum()}, seams to be no missing values in the dataset. But has Nan values in following")

def nan_vasteros(df):
    sports_feb = df[
        (df["month"] == "2024-02") &
        (df["city"] == "Västerås") &
        (df["category"] == "Sports")
    ]

    if sports_feb.empty:
        print("No data for Sports in Västerås for February 2024.")
    else:
        print("Data for Sports in Västerås for February 2024 is available.")

    toys_may= df[
        (df["month"] == "2024-05") &
        (df["city"] == "Västerås") &
        (df["category"] == "Toys")
    ]

    if toys_may.empty:
        print("No data for Toys in Västerås for May 2024.")
    else:
        print("Data for Toys in Västerås for May 2024 is available.")

print("Sales in theese categories:", nan_vasteros(df))

In [ ]:
#då ovan bara visar att det saknas försäljning över en hel månad under dessa kategorier i västerås, vill jag ta reda på om det är hela månaden eller någon specifik dag som saknas.

sports_feb_days = df[
    (df["month"] == "2024-02") &
    (df["city"] == "Västerås") &
    (df["category"] == "Sports")
]

toys_may_days = df[
    (df["month"] == "2024-05") &
    (df["city"] == "Västerås") &
    (df["category"] == "Toys")
]


print("Checking for missing days in February 2024 for Sports in Västerås:")
print(sports_feb_days[["date", "units"]].sort_values("date"))

print("Checking for missing days in May 2024 for Toys in Västerås:")
print(toys_may_days[["date", "units"]].sort_values("date"))

Ovan resultat visar att det inte finns någon registrerad försäljning alls under sports och toys i västerås under hela månadern februari och maj. 

för att visa det tydligare kommer jag visa vilka månader som västerås faktiskt har försäljning på dessa kategorier.
Det är alltså inte någon enstaka dag som saknar försäljning utan hela månaden. 

In [ ]:
#visa vilka månader som faktiskt har försäljning för sports och toys för västerås. om månad 2 och 5 saknas så är det hela månaden som saknas. 

months_with_sales_sports = sorted(
    df[
        (df["city"] == "Västerås") &
        (df["category"] == "Sports")
    ]["month"].unique()
)

print("Months with sales for Sports in Västerås:")
print(months_with_sales_sports)

months_with_sales_toys = sorted(
    df[
        (df["city"] == "Västerås") &
        (df["category"] == "Toys")
    ]["month"].unique()
)

print("Months with sales for Toys in Västerås:")
print(months_with_sales_toys)


Resultatet visar att det är månaderna 1,3,4,5 och 6 som har försäljning av sports. månad 2 (februari) saknar alltså försäljning för kategorin sports i västerås

Resultatet visar att månaderna: 1,2,3,4 och 6 har försäljning av toys. Månad 5 (maj) saknar alltså försäljning för kategorin toys i västerås


Nu när jag har hanterat Nan värderna kommer jag fortsätta att undersöka om det är någon försäljning som är ovanlig på något sätt.

In [175]:
#Jag vill börja med att undersöka högsta revenue per stad under hela perioden. För att sedan jämföra mellan städerna och sedan vad som skiljer dem mer.

max_revenue_per_city = df.loc[
    df.groupby("city")["revenue"].idxmax(),
    ["city", "category", "revenue"]
].sort_values("revenue",ascending=False)

print("Maximum revenue per city:")
print(max_revenue_per_city)

Maximum revenue per city:
           city     category  revenue
1411  Stockholm  Electronics  9192.60
1339   Göteborg  Electronics  8516.85
1491   Västerås  Electronics  8370.12
1733    Uppsala  Electronics  7366.38
2282      Malmö       Sports  6293.34


Här ser vi att Malmö skiljer deras högsta revenue kategori är Sports medans resten av städerna är Electronics. Vilket gör det intressant att undersöka vad Malmö har för högsta revenue i övriga kategorier.

In [174]:
#Malmös högsta revenue i alla kategorier.

malmo_max_revenue = (
    df[df["city"] == "Malmö"]
    .groupby("category")["revenue"]
    .max()
    .sort_values(ascending=False)
)

print("Maximum revenue in Malmö by category:")
print(malmo_max_revenue)

Maximum revenue in Malmö by category:
category
Sports         6293.34
Electronics    6260.58
Clothing       3226.80
Home           3159.80
Toys           1845.13
Beauty         1579.83
Name: revenue, dtype: float64


Här ser vi att Sports tog första platsen med liten marginal vilket gör det intressant att fortsätta att kolla på andra städers max revenue i alla kategorier och se om deras sports ligger som nr två eller hur de skiljer sig.

In [ ]:
#Undersöka vilka kategorier i alla andra städer som har högsta revenue och jämföra med malmö.
city_max_revenue = []

for city in cities:
    if city != "Malmö":
        max_revenue_category = max_revenue_category_city(df, city)
        max_revenue_category.name = city
        city_max_revenue.append(max_revenue_category)

compare_max_revenue = pd.concat(city_max_revenue, axis=1)
print("\nComparison of maximum revenue by category across cities:")
print(compare_max_revenue)


Comparison of maximum revenue by category across cities:
             Stockholm  Uppsala  Göteborg  Västerås
category                                           
Electronics    9192.60  7366.38   8516.85   8370.12
Sports         5628.18  5598.06   5455.38   6312.48
Clothing       4634.56  5480.24   3516.60   3795.40
Toys           3511.62  1532.00   1980.72   1842.70
Home           3123.24  3640.45   3926.40   3005.52
Beauty         2675.80  2497.64   2473.02   2315.65


Här ser vi att kategorin sports är nummer 2 i samtliga andra städer förutom i malmö då den var nr 1. Electronics ligger överst för alla städer förutom malmö. 

Malmös övriga kategorier skiljer sig ytterligare på kategorierna toys och home som byter plats jämfört med resterande städer. 

Malmös handlingsmönster är alltså lite annorlunda än resterande städer. 

Frågan som jag nu ställer mig är det en specifik månad som samtliga högsta revenues uppstår i eller är de olika för olika städer?

In [194]:
#undersöka när högsta revenue inträffar för samtliga städer

max_revenue_months = df.loc[
    df.groupby("city")["revenue"].idxmax(),
    ["city", "category", "revenue", "month"]
].sort_values("revenue", ascending=False)

print("Months when maximum revenue occurred for each city:")
print(max_revenue_months)

Months when maximum revenue occurred for each city:
           city     category  revenue    month
1411  Stockholm  Electronics  9192.60  2024-02
1339   Göteborg  Electronics  8516.85  2024-02
1491   Västerås  Electronics  8370.12  2024-06
1733    Uppsala  Electronics  7366.38  2024-06
2282      Malmö       Sports  6293.34  2024-06


Intressant att februari och juni sticker ut som de månaderna som högsta revenue inträffade på. Stockholm och Göteborgs sålde högst i februari och övriga städer i juni. 

Då ställer jag mig frågan hur såg stockholm och göteborg ut i juni samt hur såg övriga städer ut i februari.

In [195]:
#Undersöka hur februari såg ut för västerås, uppsala och malmö. hur juni såg ut för stockholm och göteborg.

max_revenue_february_three_cities = df.loc[
    df[
        (df["month"] == "2024-02") &
        (df["city"].isin(["Västerås", "Uppsala", "Malmö"]))
    ]
    .groupby("city")["revenue"]
    .idxmax()
]

print("Maximum revenue in February 2024 for Västerås, Uppsala, and Malmö:")
print(max_revenue_february_three_cities)

Maximum revenue in February 2024 for Västerås, Uppsala, and Malmö:
      order_id       date      city     category    price  units  revenue  \
770     100770 2024-02-23     Malmö  Electronics  1169.44      5  5847.20   
116     100116 2024-02-10   Uppsala  Electronics  1593.28      4  6373.12   
1508    101508 2024-02-06  Västerås  Electronics   750.75      5  3753.75   

        month  
770   2024-02  
116   2024-02  
1508  2024-02  


Februari är alltså den månad som electronics har högst revenue i samtliga städer. 

Hur ser nu juni ut?

In [193]:
max_revenue_juni_three_cities = df.loc[
    df[
        (df["month"] == "2024-06") &
        (df["city"].isin(["Stockholm", "Göteborg"]))
    ]
    .groupby("city")["revenue"]
    .idxmax()
]

print("Maximum revenue in Juni 2024 for Göteborg and Stockholm:")
print(max_revenue_juni_three_cities)

Maximum revenue in Juni 2024 for Göteborg and Stockholm:
      order_id       date       city     category    price  units  revenue  \
1061    101061 2024-06-09   Göteborg  Electronics  1460.44      4  5841.76   
233     100233 2024-06-17  Stockholm       Sports   675.64      6  4053.84   

        month  
1061  2024-06  
233   2024-06  


Juni har Uppsala och Göteborg högst revenue för electrolics. Stockholm och Malmö har sports som högst i juni.

Nu ställer jag mig frågan när är vilken kategori top 1 i revenue, är det samma för hela perioden eller skiljer det sig?

In [192]:
# undersöka om någon kategori dominerar hela perioden i någon stad eller om det är olika hela tiden.

# monthly_top_revenue = df.loc[
#     df.groupby(["city", "month"])["revenue"].idxmax(),
#     ["city", "month", "category"]
# ]

# print("\nMonthly top revenue categories for each city:")
# print(monthly_top_revenue)

top_winners_tab = {}

for city in monthly_top_revenue["city"].unique():

    city_data = monthly_top_revenue[
        monthly_top_revenue["city"] == city
        ]

    category_months = {}

    for category in city_data["category"].unique():
        months = sorted(
            city_data[
            city_data["category"] == category
            ]["month"]
            .str[-2:]
            .astype(int)
            .tolist()
            
        )
        def format_months(months):

            if months == list(range(min(months), max(months) + 1)):
                return f"{min(months)}-{max(months)}"

            return ",".join(map(str, months))

        category_months[category] = format_months(months)
        top_winners_tab[city] = pd.Series(category_months)

compare_top_winners = pd.DataFrame(top_winners_tab)
compare_top_winners = compare_top_winners.fillna("")

print("Comparison of top revenue categories across cities:")
print(compare_top_winners)

Comparison of top revenue categories across cities:
            Göteborg Malmö Stockholm Uppsala Västerås
Electronics      1-6   1-5   1,2,3,5     1-6  1,2,4,6
Home                                              3,5
Sports                 6-6       4,6                 


Här ser vi att Malmös högsta revenue var en engångsföreteelse. I övriga månader är electronics den högsta inkomstkällan.

Nu har jag specificerat när de största revenue topparna har hänt och från var. 
Nu ställer jag mig frågan:

När jag vet topparna, vad är egenligen medelköpet i respektive stad och kategori, något som sticker ut där?

In [197]:
#hitta medianen för samtliga städer och kategorier för att se om det finns några avvikelser.

city_median_revenue = (
    df.groupby(["city", "category"])["revenue"]
    .median()
    .unstack()
)

print("Comparison of median revenue by category across cities:")
print(city_median_revenue)

Comparison of median revenue by category across cities:
category    Beauty  Clothing  Electronics    Home    Sports    Toys
city                                                               
Göteborg   719.645  1108.790     1944.920  855.79  1933.880  588.29
Malmö      606.740   991.980     2061.395  970.71  1474.370  453.90
Stockholm  532.160  1055.640     1768.590  857.79  1519.950  525.48
Uppsala    639.560  1004.385     2000.120  912.30  1434.965  514.19
Västerås   492.840   885.140     1550.805  980.12  1440.080  514.16


Intressant med det här är att Malmös högsta revenue totalt är i sports men högsta median är i electronics.

In [202]:
monthly_median_revenue = (
    df.groupby(["city", "month", "category"])["revenue"]
    .median()
    .reset_index()
)

monthly_top_median = monthly_median_revenue.loc[
    monthly_median_revenue.groupby(
        ["city", "month"]
    )["revenue"].idxmax()
]

top_median_tab = {}

for city in monthly_median_revenue["city"].unique():

    city_data = monthly_top_median[
        monthly_top_median["city"] == city
        ]

    category_months = {}

    for category in city_data["category"].unique():
        months = sorted(
            city_data[
            city_data["category"] == category
            ]["month"]
            .str[-2:]
            .astype(int)
            .tolist()
            
        )
        def format_months(months):

            if months == list(range(min(months), max(months) + 1)):
                return f"{min(months)}-{max(months)}"

            return ",".join(map(str, months))

        category_months[category] = format_months(months)
        top_median_tab[city] = pd.Series(category_months)

compare_median_winners = pd.DataFrame(top_winners_tab)
compare_median_winners = compare_median_winners.fillna("")

print("Comparison of median revenue categories across cities:")
print(compare_median_winners)

Comparison of median revenue categories across cities:
            Göteborg Malmö Stockholm Uppsala Västerås
Clothing                                          4-4
Electronics      1-3   1-3   1,2,3,6   1,2,4      1-3
Sports           4-6   4-6       4-5   3,5,6      5-6


Här ser vi något som sticker ut. Clothing har aldrig varit nämnt på revenu lista någonstans. Ändå är det högst medelvärde på den kategorin i april för västerås. 

Övriga månader ser vi fortsatt electronics och sport som dominerar både top revenue och median revenue. 

Nu ställer jag mig frågan:
Hur såg april ut för västerås i resterande kategorier?


In [212]:
#ta fram median och top för västerås i april

vasteras_april_med_max = (
    df[
        (df["city"] == "Västerås") &
        (df["month"] == "2024-04")
    ]
    .groupby("category")
    .agg(
        Median = ("revenue", "median"),
        Max =  ("revenue", "max"),
        Units = ("units", "sum"),
        Orders = ("order_id", "count")
        )
    .sort_values("Median", ascending=False)
)

print("Median and Max revenue in Västerås for April 2024 by category:")
print(vasteras_april_med_max)

Median and Max revenue in Västerås for April 2024 by category:
               Median      Max  Units  Orders
category                                     
Clothing     1202.900  2568.96     53      15
Electronics  1188.840  5175.36      9       5
Sports        980.600   980.60      4       1
Home          876.540  3005.52     28      11
Beauty        492.840  1713.55     17       7
Toys          467.835  1292.60     19       6


Här ser vi att högsta enskilda köpen ligger under electronics och home. Men att det största antalet ordrar och varor hamnar under clothing vilket gör att totala medelvärdet blir högre. 

Jag ställer mig nu frågan:
Är clothing alltid den kategori med flest orders och units eller är detta ett enstaka fall?

In [227]:
#Tabeller för antal orders och units för samtliga städer och kategorier under hela perioden.

units_pivot = (
    df.groupby(["category", "city"])
    ["units"]
    .sum()
    .unstack()
)

orders_pivot = (
    df.groupby(["category", "city"])
    ["order_id"]
    .count()
    .unstack()
)

print("Total Units by City and Category:")
print(units_pivot)

print("\nTotal Orders by City and Category:")
print(orders_pivot)

Total Units by City and Category:
city         Göteborg  Malmö  Stockholm  Uppsala  Västerås
category                                                  
Beauty            162     95        223       99        80
Clothing          338    228        574      303       212
Electronics       425    313        599      293       123
Home              364    215        419      171       109
Sports            341    247        539      178        96
Toys              158    113        265      121        60

Total Orders by City and Category:
city         Göteborg  Malmö  Stockholm  Uppsala  Västerås
category                                                  
Beauty             56     31         86       37        31
Clothing          104     69        181       86        69
Electronics       145    104        220      101        48
Home              136     83        163       65        43
Sports             91     70        169       56        29
Toys               54     39         80      

För västerås är clothing den kategori som flest köper och med störst antal. Under april månad tog västerås in en stor del av totala andelen ordrar men bara en liten del av antal produkter.

Ser vi på hela perioden och alla städer så är clothing INTE en kategori som sticker ut vad gäller antal ordrar eller produkter. Den är med i toppen för västerås men varierar jämfört med resterande städer.

Att västerås fick en topp med clothing i april anses vara ett engångsfall.